# Exercises

These are PHREEQ exersices from Dr Weiss's undergrad classes. They demonstrate PHREEQ capabilities and usage. I'll be using them to validate the accuracy of PHREEQpy to ensure that I've installed it correcly and am using the right way

## Exercise 1

In [29]:
import phreeqpy.iphreeqc.phreeqc_dll as phreeqc_mod
import pandas as pd
import numpy as np

The phreeqpy library is a bit touchy with importing the c++ libs and database files. I'll have to workshop some solutions. The code below is justa bandaid.

In [30]:
database = "../llnl.dat"

In [31]:
def get_results(freak):
    """Extracts simulation info from a phreeqc instance"""
    output = freak.get_selected_output_array()
    results = pd.DataFrame(output[1:], columns=output[0])
    results.index = results.index + 1
    results.index.name = "solution"
    return results

Make input file

In [32]:
input_1 = """
TITLE 02_pH_DEPENDENCE

SOLUTION_MASTER_SPECIES
As(+5)	H2AsO4-		0.0	As	74.92
END

SOLUTION_SPECIES
AsO4-3 + H+ = HAsO4-2
log_k		11.60
AsO4-3 + 2H+ = H2AsO4-
log_k		18.35
AsO4-3 + 3H+ = H3AsO4
log_k		20.60
END

SELECTED_OUTPUT
-file			output.xls
-reset		false
-high_precision false
-molalities		AsO4-3   HAsO4-2   H2AsO4-   H3AsO4
END

SOLUTION 1
pH		3.0
Cl		0.1		mol/kgw
Na		0.1		mol/kgw
As(+5)	100		ug/kgw
END

SOLUTION 2
pH		4.0
Cl		0.1		mol/kgw
Na		0.1		mol/kgw
As(+5)	100		ug/kgw
END

SOLUTION 3
pH		5.0
Cl		0.1		mol/kgw
Na		0.1		mol/kgw
As(+5)	100		ug/kgw
END

SOLUTION 4
pH		6.0
Cl		0.1		mol/kgw
Na		0.1		mol/kgw
As(+5)	100		ug/kgw
END

SOLUTION 5
pH		7.0
Cl		0.1		mol/kgw
Na		0.1		mol/kgw
As(+5)	100		ug/kgw
END

SOLUTION 6
pH		8.0
Cl		0.1		mol/kgw
Na		0.1		mol/kgw
As(+5)	100		ug/kgw
END

SOLUTION 7
pH		9.0
Cl		0.1		mol/kgw
Na		0.1		mol/kgw
As(+5)	100		ug/kgw
END

SOLUTION 8
pH		10.0
Cl		0.1		mol/kgw
Na		0.1		mol/kgw
As(+5)	100		ug/kgw
END

SOLUTION 9
pH		11.0
Cl		0.1		mol/kgw
Na		0.1		mol/kgw
As(+5)	100		ug/kgw
END
"""

Initialize PHREEQC from IPhreeqc

In [33]:
# freak = phreeqc_mod.IPhreeqc(IPhreeqc_lib)
freak = phreeqc_mod.IPhreeqc()
freak.load_database(database)
freak.run_string(input_1)

In [34]:
results = get_results(freak)
freak.destroy_iphreeqc()
results.head()

,m_AsO4-3(mol/kgw),m_HAsO4-2(mol/kgw),m_H2AsO4-(mol/kgw),m_H3AsO4(mol/kgw)
solution,,,,
1,5.132324e-18,1.940855e-10,1.144848e-06,1.896861e-07
2,5.850214e-16,2.222168e-09,1.310786e-06,2.172064e-08
3,5.845640e-14,2.221417e-08,1.310343e-06,2.171357e-09
4,5.090347e-12,1.934478e-07,1.141087e-06,1.890886e-10
5,2.208730e-10,8.393776e-07,4.951219e-07,8.204628e-12


lets compare this to the results that were given in the answer key

In [35]:
answers = pd.read_csv('anser_key/exercise_1_concentrations.csv')
answers = answers.iloc[:, -4:]
answers.index = results.index
answers.columns = results.columns
answers.head()

,m_AsO4-3(mol/kgw),m_HAsO4-2(mol/kgw),m_H2AsO4-(mol/kgw),m_H3AsO4(mol/kgw)
solution,,,,
1,5.130000e-18,1.940000e-10,1.140000e-06,1.900000e-07
2,5.850000e-16,2.220000e-09,1.310000e-06,2.170000e-08
3,5.850000e-14,2.220000e-08,1.310000e-06,2.170000e-09
4,5.090000e-12,1.930000e-07,1.140000e-06,1.890000e-10
5,2.210000e-10,8.390000e-07,4.950000e-07,8.200000e-12


In [36]:
def check_dataframes(df1, df2):
    """Test of two dataframes are equal"""
    assert df1.shape == df2.shape
    assert df1.columns.equals(df2.columns)
    assert df1.index.equals(df2.index)
    assert np.allclose(df1, df2)

In [37]:
check_dataframes(results, answers)

## Exercise 3

input file

In [38]:
input_3 = """
TITLE 03_SURFACE_ACIDITY

SOLUTION_MASTER_SPECIES
As(+5)	H2AsO4-		0.0	As	74.92
END

SOLUTION_SPECIES
AsO4-3 + H+ = HAsO4-2
log_k		11.60
AsO4-3 + 2H+ = H2AsO4-
log_k		18.35
AsO4-3 + 3H+ = H3AsO4
log_k		20.60

SURFACE_MASTER_SPECIES 
Surf_ Surf_OH

SURFACE_SPECIES
Surf_OH = Surf_OH   						 #goethite surface protonation reactions 
log_k		0.0
Surf_OH + H+ = Surf_OH2+
log_k		7.47
Surf_OH = Surf_O- + H+
log_k		-9.51            

Surf_OH + AsO4-3 + 3H+ = Surf_H2AsO4 + H2O		#goethite surface complexation reactions
log_k		31.00
Surf_OH + AsO4-3 + 2H+ = Surf_HAsO4- + H2O
log_k		26.81
Surf_OH + AsO4-3 + H+ = Surf_AsO4-2 + H2O
log_k		20.22
END

SURFACE 1 
      -sites DENSITY	
	Surf_OH		2.0     54     0.5		#name of binding site, (sites/site density), specific area per gram, grams
END

SELECTED_OUTPUT
-reset		true
-molalities		Surf_OH   Surf_OH2+   Surf_O-
END

SOLUTION 1
pH		3.0
Cl		0.1		mol/kgw
Na		0.1		mol/kgw
As(+5)	0.000 	mmol/kgw
END

PHASES 
     Fix_H+ 
     H+ = H+ 
     log_k  0.0 		#pseudo-
END 

USE SURFACE 1
USE SOLUTION 1
EQUILIBRIUM_PHASES 1
	Fix_H+ -3.0 NaOH 11.0 			#adds HNO3 as the Fix_H+ pseudo-phase to obtain a pH of exactly 6
END

USE SURFACE 1
USE SOLUTION 1
EQUILIBRIUM_PHASES 1
	Fix_H+ -3.5 NaOH 11.0 			#adds HNO3 as the Fix_H+ pseudo-phase to obtain a pH of exactly 6
END

USE SURFACE 1
USE SOLUTION 1
EQUILIBRIUM_PHASES 1
	Fix_H+ -4.0 NaOH 11.0 			#adds HNO3 as the Fix_H+ pseudo-phase to obtain a pH of exactly 6
END

USE SURFACE 1
USE SOLUTION 1
EQUILIBRIUM_PHASES 1
	Fix_H+ -4.5 NaOH 11.0 			#adds HNO3 as the Fix_H+ pseudo-phase to obtain a pH of exactly 6
END

USE SURFACE 1
USE SOLUTION 1
EQUILIBRIUM_PHASES 1
	Fix_H+ -5.0 NaOH 11.0 			#adds HNO3 as the Fix_H+ pseudo-phase to obtain a pH of exactly 6
END

USE SURFACE 1
USE SOLUTION 1
EQUILIBRIUM_PHASES 1
	Fix_H+ -5.5 NaOH 11.0 			#adds HNO3 as the Fix_H+ pseudo-phase to obtain a pH of exactly 6
END

USE SURFACE 1
USE SOLUTION 1
EQUILIBRIUM_PHASES 1
	Fix_H+ -6.0 NaOH 11.0 			#adds HNO3 as the Fix_H+ pseudo-phase to obtain a pH of exactly 6
END

USE SURFACE 1
USE SOLUTION 1
EQUILIBRIUM_PHASES 1
	Fix_H+ -6.5 NaOH 11.0 			#adds HNO3 as the Fix_H+ pseudo-phase to obtain a pH of exactly 6
END

USE SURFACE 1
USE SOLUTION 1
EQUILIBRIUM_PHASES 1
	Fix_H+ -7.0 NaOH 11.0 			#adds HNO3 as the Fix_H+ pseudo-phase to obtain a pH of exactly 6
END

USE SURFACE 1
USE SOLUTION 1
EQUILIBRIUM_PHASES 1
	Fix_H+ -7.5 NaOH 11.0 			#adds HNO3 as the Fix_H+ pseudo-phase to obtain a pH of exactly 6
END

USE SURFACE 1
USE SOLUTION 1
EQUILIBRIUM_PHASES 1
	Fix_H+ -8.0 NaOH 11.0 			#adds HNO3 as the Fix_H+ pseudo-phase to obtain a pH of exactly 6
END

USE SURFACE 1
USE SOLUTION 1
EQUILIBRIUM_PHASES 1
	Fix_H+ -8.5 NaOH 11.0 			#adds HNO3 as the Fix_H+ pseudo-phase to obtain a pH of exactly 6
END

USE SURFACE 1
USE SOLUTION 1
EQUILIBRIUM_PHASES 1
	Fix_H+ -9.0 NaOH 11.0 			#adds HNO3 as the Fix_H+ pseudo-phase to obtain a pH of exactly 6
END

USE SURFACE 1
USE SOLUTION 1
EQUILIBRIUM_PHASES 1
	Fix_H+ -9.5 NaOH 11.0 			#adds HNO3 as the Fix_H+ pseudo-phase to obtain a pH of exactly 6
END

USE SURFACE 1
USE SOLUTION 1
EQUILIBRIUM_PHASES 1
	Fix_H+ -10.0 NaOH 11.0 			#adds HNO3 as the Fix_H+ pseudo-phase to obtain a pH of exactly 6
END

USE SURFACE 1
USE SOLUTION 1
EQUILIBRIUM_PHASES 1
	Fix_H+ -10.5 NaOH 11.0 			#adds HNO3 as the Fix_H+ pseudo-phase to obtain a pH of exactly 6
END

USE SURFACE 1
USE SOLUTION 1
EQUILIBRIUM_PHASES 1
	Fix_H+ -11.0 NaOH 11.0 			#adds HNO3 as the Fix_H+ pseudo-phase to obtain a pH of exactly 6
END
"""

build simulation, extract results, and destroy model for space

In [39]:
freak = phreeqc_mod.IPhreeqc()
freak.load_database(database)
freak.run_string(input_3)

results = get_results(freak)
freak.destroy_iphreeqc()

inspect results

In [40]:
results.head()

,sim,state,soln,dist_x,time,step,pH,pe,reaction,temp(C),Alk(eq/kgw),mu,mass_H2O,charge(eq),pct_err,m_Surf_OH(mol/kgw),m_Surf_OH2+(mol/kgw),m_Surf_O-(mol/kgw)
solution,,,,,,,,,,,,,,,,,,
1,5,i_soln,1,-99.0,-99.0,-99,3.0,4.000000,-99,25.0,-0.001217,0.099576,1.000000,0.001217,0.611303,0.000000e+00,0.000000,0.000000e+00
2,7,react,1,-99.0,0.0,1,3.0,15.218306,-99,25.0,-0.001217,0.099532,0.999998,0.001129,0.566974,8.887827e-07,0.000089,8.115247e-11
3,8,react,1,-99.0,0.0,1,3.5,14.779649,-99,25.0,-0.000385,0.099535,1.000013,0.001130,0.567836,2.647359e-06,0.000087,7.345741e-10
4,9,react,1,-99.0,0.0,1,4.0,14.349949,-99,25.0,-0.000122,0.099537,1.000018,0.001135,0.570081,7.143191e-06,0.000083,5.639779e-09
5,10,react,1,-99.0,0.0,1,4.5,13.931736,-99,25.0,-0.000038,0.099542,1.000020,0.001144,0.574551,1.604248e-05,0.000074,3.189629e-08


import solution results

In [41]:
a = pd.read_csv('anser_key/exercise_3.csv')
a.index = results.index
a.columns = results.columns
a.head()
results.drop('state', axis=1, inplace=True)
a.drop('state', axis=1, inplace=True)


In [42]:
results.head()

,sim,soln,dist_x,time,step,pH,pe,reaction,temp(C),Alk(eq/kgw),mu,mass_H2O,charge(eq),pct_err,m_Surf_OH(mol/kgw),m_Surf_OH2+(mol/kgw),m_Surf_O-(mol/kgw)
solution,,,,,,,,,,,,,,,,,
1,5,1,-99.0,-99.0,-99,3.0,4.000000,-99,25.0,-0.001217,0.099576,1.000000,0.001217,0.611303,0.000000e+00,0.000000,0.000000e+00
2,7,1,-99.0,0.0,1,3.0,15.218306,-99,25.0,-0.001217,0.099532,0.999998,0.001129,0.566974,8.887827e-07,0.000089,8.115247e-11
3,8,1,-99.0,0.0,1,3.5,14.779649,-99,25.0,-0.000385,0.099535,1.000013,0.001130,0.567836,2.647359e-06,0.000087,7.345741e-10
4,9,1,-99.0,0.0,1,4.0,14.349949,-99,25.0,-0.000122,0.099537,1.000018,0.001135,0.570081,7.143191e-06,0.000083,5.639779e-09
5,10,1,-99.0,0.0,1,4.5,13.931736,-99,25.0,-0.000038,0.099542,1.000020,0.001144,0.574551,1.604248e-05,0.000074,3.189629e-08


check to make sure that results are the same

In [43]:
print(np.allclose(results, a, atol=1e-2))

False


there is a difference somewhere. Lets find different values

In [44]:
comparison = a.loc[:, 'pe'].compare(results.loc[:, 'pe'])
diff = comparison['self'] - comparison['other']
print(diff)


solution
2     0.000094
3    -0.000049
4    -0.000049
5    -0.000436
6     0.255925
7     0.000313
8    -0.000214
9     0.001984
10   -0.001060
11    0.000595
12    0.001394
13   -0.106246
14   -0.000831
15   -0.000560
16    0.002001
17    0.001026
18    0.004791
dtype: float64
